In [33]:
# Libraries
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Lipinski
from sklearn.cluster import DBSCAN
import numpy as np

In [34]:
# Define the SMILES strings for the flavonoids
dicc_smiles = {
    'Tricetin' : 'O=c1cc(-c2cc(O)c(O)c(O)c2)oc2cc(O)cc(O)c12',
    'Scutellarein' : 'O=c1cc(-c2ccc(O)cc2)oc2cc(O)c(O)c(O)c12',
    'Baicalein' : 'O=c1cc(-c2ccccc2)oc2cc(O)c(O)c(O)c12',
    'Robinetin' : 'O=c1c(O)c(-c2cc(O)c(O)c(O)c2)oc2cc(O)ccc12',
    'Myricetin' : 'O=c1c(O)c(-c2cc(O)c(O)c(O)c2)oc2cc(O)cc(O)c12',
    'Baicalin' : 'O=C(O)[C@H]1O[C@@H](Oc2cc3oc(-c4ccccc4)cc(=O)c3c(O)c2O)[C@H](O)[C@@H](O)[C@@H]1O',
    'Chrysin' : 'O=c1cc(-c2ccccc2)oc2cc(O)cc(O)c12',
    'Isoorientin' : 'O=c1cc(-c2ccc(O)c(O)c2)oc2cc(O)c([C@@H]3O[C@H](CO)[C@@H](O)[C@H](O)[C@H]3O)c(O)c12',
    'Phlorizin' : 'O=C(CCc1ccc(O)cc1)c1c(O)cc(O)cc1O[C@@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@H]1O',
    'Farrerol' : 'Cc1c(O)c(C)c2c(c1O)C(=O)C[C@@H](c1ccc(O)cc1)O2'
}

In [35]:
df_remarks_top_pct = pd.read_csv('df_remarks_top_pct.csv')

In [36]:
# Create a dictionary to store the pharmacophore features for each ligand
dicc_ligand_pharmacophore = {}

for ligand in df_remarks_top_pct['ligand'].unique():
    smiles = dicc_smiles[ligand]
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)

    hbd_atoms = set()
    for match in mol.GetSubstructMatches(Lipinski.HDonorSmarts):
        hbd_atoms.add(match[0])
    dicc_ligand_pharmacophore[ligand + '_hbd'] = hbd_atoms

    hba_atoms = set()
    for match in mol.GetSubstructMatches(Lipinski.HAcceptorSmarts):
        hba_atoms.add(match[0])
    dicc_ligand_pharmacophore[ligand + '_hba'] = hba_atoms

    aromatic_rings = []
    ring_info = mol.GetRingInfo()

    for ring in ring_info.AtomRings():
        if all(mol.GetAtomWithIdx(i).GetIsAromatic() for i in ring):
            aromatic_rings.append(ring)
    dicc_ligand_pharmacophore[ligand + '_arom_rings'] = aromatic_rings

    print(ligand)
    print("HBD atoms:", sorted(hbd_atoms))
    print("HBA atoms:", sorted(hba_atoms))
    print("Aromatic rings:")
    for i, ring in enumerate(aromatic_rings):
        print(f"  Ring {i}: {ring}")

Phlorizin
HBD atoms: [8, 13, 16, 24, 26, 28, 30]
HBA atoms: [0, 8, 13, 16, 19, 21, 24, 26, 28, 30]
Aromatic rings:
  Ring 0: (5, 6, 7, 9, 10, 4)
  Ring 1: (12, 14, 15, 17, 18, 11)
Baicalin
HBD atoms: [2, 23, 25, 27, 29, 31]
HBA atoms: [0, 2, 4, 6, 10, 20, 23, 25, 27, 29, 31]
Aromatic rings:
  Ring 0: (8, 9, 21, 22, 24, 7)
  Ring 1: (10, 11, 18, 19, 21, 9)
  Ring 2: (13, 14, 15, 16, 17, 12)
Isoorientin
HBD atoms: [8, 10, 16, 22, 24, 26, 28, 30]
HBA atoms: [0, 8, 10, 12, 16, 19, 22, 24, 26, 28, 30]
Aromatic rings:
  Ring 0: (1, 31, 13, 12, 3, 2)
  Ring 1: (5, 6, 7, 9, 11, 4)
  Ring 2: (14, 15, 17, 29, 31, 13)
Myricetin
HBD atoms: [3, 8, 10, 12, 18, 21]
HBA atoms: [0, 3, 8, 10, 12, 14, 18, 21]
Aromatic rings:
  Ring 0: (1, 22, 15, 14, 4, 2)
  Ring 1: (6, 7, 9, 11, 13, 5)
  Ring 2: (16, 17, 19, 20, 22, 15)
Robinetin
HBD atoms: [3, 8, 10, 12, 18]
HBA atoms: [0, 3, 8, 10, 12, 14, 18]
Aromatic rings:
  Ring 0: (1, 21, 15, 14, 4, 2)
  Ring 1: (6, 7, 9, 11, 13, 5)
  Ring 2: (16, 17, 19, 20, 21,

In [37]:
# Display atom indices and connectivity for manual pharmacophore annotation
for ligand in df_remarks_top_pct['ligand'].unique():

    smiles = dicc_smiles[ligand]
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)

    print(ligand)

    print("Idx\tElement\tNeighbors")
    for atom in mol.GetAtoms():
        idx = atom.GetIdx()
        symbol = atom.GetSymbol()
        neighbors = [nbr.GetIdx() for nbr in atom.GetNeighbors()]
        print(f"{idx}\t{symbol}\t{neighbors}")

Phlorizin
Idx	Element	Neighbors
0	O	[1]
1	C	[0, 2, 11]
2	C	[1, 3, 31, 32]
3	C	[2, 4, 33, 34]
4	C	[3, 5, 10]
5	C	[4, 6, 35]
6	C	[5, 7, 36]
7	C	[6, 8, 9]
8	O	[7, 37]
9	C	[7, 10, 38]
10	C	[9, 4, 39]
11	C	[1, 12, 18]
12	C	[11, 13, 14]
13	O	[12, 40]
14	C	[12, 15, 41]
15	C	[14, 16, 17]
16	O	[15, 42]
17	C	[15, 18, 43]
18	C	[17, 19, 11]
19	O	[18, 20]
20	C	[19, 21, 29, 44]
21	O	[20, 22]
22	C	[21, 23, 25, 45]
23	C	[22, 24, 46, 47]
24	O	[23, 48]
25	C	[22, 26, 27, 49]
26	O	[25, 50]
27	C	[25, 28, 29, 51]
28	O	[27, 52]
29	C	[27, 30, 20, 53]
30	O	[29, 54]
31	H	[2]
32	H	[2]
33	H	[3]
34	H	[3]
35	H	[5]
36	H	[6]
37	H	[8]
38	H	[9]
39	H	[10]
40	H	[13]
41	H	[14]
42	H	[16]
43	H	[17]
44	H	[20]
45	H	[22]
46	H	[23]
47	H	[23]
48	H	[24]
49	H	[25]
50	H	[26]
51	H	[27]
52	H	[28]
53	H	[29]
54	H	[30]
Baicalin
Idx	Element	Neighbors
0	O	[1]
1	C	[0, 2, 3]
2	O	[1, 32]
3	C	[1, 4, 30, 33]
4	O	[3, 5]
5	C	[4, 6, 26, 34]
6	O	[5, 7]
7	C	[6, 8, 24]
8	C	[7, 9, 35]
9	C	[8, 10, 21]
10	O	[9, 11]
11	C	[10, 12, 18]
12	C	[11, 13, 17]
1

In [38]:
# Manual assignment of pharmacophore features based on RDKit atom connectivity and PyMOL inspection
dicc_phlorizin = {
  'hbd' : ['O7', 'O5', 'O4', 'O1', 'O10', 'O9', 'O8'],
  'hba' : ['O6', 'O7', 'O5', 'O4', 'O3', 'O2', 'O1', 'O10', 'O9', 'O8'],
  'arom_rings' : ['C13_C14_C15_C16_C17_C18', 'C4_C5_C6_C7_C8_C9']
}

dicc_baicalin = {
  'hbd' : ['O10', 'O5', 'O4', 'O2', 'O1', 'O9'],
  'hba' : ['O11', 'O10', 'O8', 'O3', 'O7', 'O6', 'O5', 'O4', 'O2', 'O1', 'O9'],
  'arom_rings' : ['C4_C12_C11_C7_C6_C5', 'O7_C10_C9_C8_C7_C11', 'C13_C14_C15_C16_C17_C18']
}

dicc_isoorientin = {
  'hbd' : ['O10', 'O11', 'O6', 'O1', 'O5', 'O4', 'O3', 'O9'],
  'hba' : ['O8', 'O10', 'O11', 'O7', 'O6', 'O2', 'O1', 'O5', 'O4', 'O3', 'O9'],
  'arom_rings' : ['C13_C12_C11_O7_C10_C14', 'C16_C17_C18_C19_C20_C21', 'C9_C8_C7_C15_C14_C10']
}

dicc_myricetin = {
  'hbd' : ['O4', 'O6', 'O7', 'O8', 'O1', 'O2'],
  'hba' : ['O5', 'O4', 'O6', 'O7', 'O8', 'O3', 'O1', 'O2'],
  'arom_rings' : ['C8_C7_C6_C5_C4_O3', 'C9_C10_C1_C12_C13_C14', 'C5_C15_C1_C2_C3_C4']
}

dicc_robinetin = {
  'hbd' : ['O3', 'O5', 'O6', 'O7', 'O1'],
  'hba' : ['O4', 'O3', 'O5', 'O6', 'O7', 'O2', 'O1'],
  'arom_rings' : ['C8_C4_C6_O2_C5_C4', 'C9_C10_C11_C12_C13_C14', 'C5_C15_C1_C2_C3_C4']
}

dicc_tricetin = {
  'hbd' : ['O5', 'O6', 'O7', 'O1', 'O2'],
  'hba' : ['O3', 'O5', 'O6', 'O7', 'O4', 'O1', 'O2'],
  'arom_rings' : ['C5_C6_C7_O4_C8_C4', 'C10_C11_C12_C13_C14_C15', 'C8_C4_C9_C1_C2_C3']
}

dicc_farrerol = {
  'hbd' : ['O1', 'O4', 'O5'],
  'hba' : ['O1', 'O4', 'O3', 'O5', 'O2'],
  'arom_rings' : ['C2_C3_C4_C6_C10_C11', 'C12_C13_C14_C15_C16_C17']
}

In [39]:
dicc_ligand_props = {
    'Phlorizin': dicc_phlorizin,
    'Baicalin': dicc_baicalin,
    'Isoorientin': dicc_isoorientin,
    'Myricetin': dicc_myricetin,
    'Robinetin': dicc_robinetin,
    'Tricetin': dicc_tricetin,
    'Farrerol': dicc_farrerol
}

In [40]:
# Functions to check if an atom is a hydrogen bond donor (HBD) or acceptor (HBA) based on the ligand properties
def is_hbd(row):
    ligand = row['ligand']
    atom = row['atom_name']
    return atom in dicc_ligand_props[ligand]['hbd']

def is_hba(row):
    ligand = row['ligand']
    atom = row['atom_name']
    return atom in dicc_ligand_props[ligand]['hba']

# Function to determine which aromatic ring an atom belongs to based on the ligand properties
def get_aromatic_ring(atom_name, ligand):
    rings = dicc_ligand_props[ligand]['arom_rings']
    
    for i, ring in enumerate(rings, start=1):
        atoms = ring.split('_')
        if atom_name in atoms:
            return f'ring_{i}'
    
    return None

In [41]:
# Getting the coordinates of the top 15% docking poses
df_all_coords = pd.read_csv('df_all_coords.csv')
df_coords_top_pct = df_all_coords.merge(
    df_remarks_top_pct[['ligand', 'frame']],
    on=['ligand', 'frame'],
    how='inner'
)

In [42]:
# Assign pharmacophore features to each atom
df_coords_top_pct['hbd'] = df_coords_top_pct.apply(is_hbd, axis=1)
df_coords_top_pct['hba'] = df_coords_top_pct.apply(is_hba, axis=1)
df_coords_top_pct['arom_ring'] = df_coords_top_pct.apply(
    lambda r: get_aromatic_ring(r['atom_name'], r['ligand']),
    axis=1
)

df_hbd = df_coords_top_pct[df_coords_top_pct["hbd"] == True].copy()
df_hba = df_coords_top_pct[df_coords_top_pct["hba"] == True].copy()
df_arom = df_coords_top_pct[df_coords_top_pct["arom_ring"].notna()].copy()

In [43]:
# Perform DBSCAN clustering on the hydrogen bond donor coordinates
coords = df_hbd[["x", "y", "z"]].to_numpy()

db = DBSCAN(
    eps=0.8,
    min_samples=10
).fit(coords)

df_hbd["cluster"] = db.labels_

hbd_cluster_stats = (
    df_hbd[df_hbd["cluster"] != -1]
    .groupby("cluster")["ligand"]
    .nunique()
    .sort_values(ascending=False)
)

sorted(df_hbd["cluster"].unique())
print(df_hbd["cluster"].value_counts())
print(hbd_cluster_stats)

cluster
 1     528
-1     373
 7     254
 14    236
 15     98
 0      60
 6      47
 4      33
 8      32
 12     21
 2      17
 16     17
 3      17
 17     16
 10     14
 9      13
 11     12
 18     12
 5      11
 19     10
 13      7
Name: count, dtype: int64
cluster
1     7
14    7
7     6
15    6
4     4
16    3
0     3
2     3
11    3
9     3
6     3
8     3
13    3
17    3
12    3
3     2
10    2
5     2
18    2
19    2
Name: ligand, dtype: int64


In [44]:
# Identify possible clusters with more than 3 points
possible_clusters = hbd_cluster_stats[hbd_cluster_stats > 3].index
for cluster in possible_clusters:
    df_c = df_hbd[df_hbd["cluster"] == cluster]

    center = df_c[["x", "y", "z"]].mean().values

    distances = np.linalg.norm(
        df_c[["x","y","z"]].values - center,
        axis=1
    )

    radius = distances.max()
    if radius < 3:
        print(f"Cluster {cluster}")
        print("  nº puntos:", len(df_c))
        print("  ligandos:", df_c["ligand"].nunique())
        print("  centro:", center)
        print("  radio:", radius)

Cluster 15
  nº puntos: 98
  ligandos: 6
  centro: [ -8.86506122 -15.9915102   30.02297959]
  radio: 1.4602402651002622
Cluster 4
  nº puntos: 33
  ligandos: 4
  centro: [ -3.30566667 -25.89684848  31.45866667]
  radio: 1.4978105241956285


In [45]:
# Perform DBSCAN clustering on the hydrogen bond acceptor coordinates
coords = df_hba[["x", "y", "z"]].to_numpy()

db = DBSCAN(
    eps=0.8,
    min_samples=10
).fit(coords)

df_hba["cluster"] = db.labels_

hba_cluster_stats = (
    df_hba[df_hba["cluster"] != -1]
    .groupby("cluster")["ligand"]
    .nunique()
    .sort_values(ascending=False)
)

sorted(df_hba["cluster"].unique())
print(df_hba["cluster"].value_counts())
print(hba_cluster_stats)

cluster
 0    1967
-1     276
 7     237
 8     106
 3      47
 4      25
 5      23
 1      20
 9      17
 6      12
 2      11
Name: count, dtype: int64
cluster
0    7
7    7
8    6
1    3
3    3
4    3
6    3
5    3
9    3
2    2
Name: ligand, dtype: int64


In [46]:
# Identify possible clusters with more than 3 points
possible_clusters = hba_cluster_stats[hba_cluster_stats > 3].index
for cluster in possible_clusters:
    df_c = df_hba[df_hba["cluster"] == cluster]

    center = df_c[["x", "y", "z"]].mean().values

    distances = np.linalg.norm(
        df_c[["x","y","z"]].values - center,
        axis=1
    )

    radius = distances.max()

    if radius < 3:

        print(f"Cluster {cluster}")
        print("  nº puntos:", len(df_c))
        print("  ligandos:", df_c["ligand"].nunique())
        print("  centro:", center)
        print("  radio:", radius)

Cluster 8
  nº puntos: 106
  ligandos: 6
  centro: [ -8.86274528 -15.95512264  29.95354717]
  radio: 1.2288161744692523


In [47]:
# Extract the coordinates of aromatic rings
df_arom_centers = (
    df_arom
    .groupby(["ligand", "frame", "arom_ring"])[["x", "y", "z"]]
    .mean()
    .reset_index()
)

coords = df_arom_centers[["x","y","z"]].to_numpy()

# Perform DBSCAN clustering on the aromatic ring coordinates
db = DBSCAN(
    eps=0.8,
    min_samples=10
).fit(coords)

df_arom_centers["cluster"] = db.labels_

arom_cluster_stats = (
    df_arom_centers[df_arom_centers["cluster"] != -1]
    .groupby("cluster")["ligand"]
    .nunique()
    .sort_values(ascending=False)
)

sorted(df_arom_centers["cluster"].unique())
print(df_arom_centers["cluster"].value_counts())
print(arom_cluster_stats)

cluster
 0    397
 1    164
-1    113
 2     39
 3      8
 4      8
Name: count, dtype: int64
cluster
0    7
1    7
2    3
4    2
3    1
Name: ligand, dtype: int64


In [48]:
# Identify possible clusters with more than 3 points
possible_clusters = arom_cluster_stats[arom_cluster_stats > 3].index
for cluster in possible_clusters:
    df_c = df_arom_centers[df_arom_centers["cluster"] == cluster]

    center = df_c[["x", "y", "z"]].mean().values

    distances = np.linalg.norm(
        df_c[["x","y","z"]].values - center,
        axis=1
    )

    radius = distances.max()

    if radius < 3:

        print(f"Cluster {cluster}")
        print("  nº puntos:", len(df_c))
        print("  ligandos:", df_c["ligand"].nunique())
        print("  centro:", center)
        print("  radio:", radius)

Cluster 1
  nº puntos: 164
  ligandos: 7
  centro: [ -9.93380183 -18.10303354  31.52602846]
  radio: 2.8921110194096737
